In [ ]:
#!pip install langchain-teddynote
# Pydantic Output Parser

In [3]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("RAG study test")

llm = ChatOpenAI(temperature=0, model="gpt-5.6-luna")

LangSmith 추적을 시작합니다.
[프로젝트명]
RAG study test


In [4]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. ...다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 
귀사 사무실에서 만나 이야기를 나눌 수 있을까요?
...
김철수
상무이사
바이크코퍼레이션
"""

In [5]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요. \n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0, model="gpt-5.6-luna")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

- **발신자:** 김철수 상무이사 (바이크코퍼레이션, chulsoo.kim@bikecorporation.me)
- **수신자:** 이은채 대리 (Teddy International, eunchae@teddyinternational.me)
- **주제:** “ZENESIS” 자전거 유통 협력 및 미팅 일정 제안
- **주요 내용:** ZENESIS 자전거의 유통 협력 논의를 위해 미팅 제안
- **제안 일정:** 다음 주 화요일, **1월 15일 오전 10시**
- **장소:** 귀사 사무실(이은채 대리 측 사무실)
- **요청 사항:** 미팅 가능 여부에 대한 회신 필요

In [6]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [7]:
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [8]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [9]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the follow questions in Korean.

Question:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

In [10]:
prompt = prompt.partial(format=parser.get_format_instructions())
prompt

PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [11]:
chain = prompt | llm

response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요. "
    }
)

output = stream_response(response, return_output=True)

{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션의 김철수 상무가 ZENESIS 자전거 유통 협력 논의를 위해 미팅을 제안했습니다. 미팅은 귀사 사무실에서 진행하기를 희망합니다.",
  "date": "다음 주 화요일(1월 15일) 오전 10시"
}

In [12]:
structed_output = parser.parse(output)
print(structed_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션의 김철수 상무가 ZENESIS 자전거 유통 협력 논의를 위해 미팅을 제안했습니다. 미팅은 귀사 사무실에서 진행하기를 희망합니다.' date='다음 주 화요일(1월 15일) 오전 10시'


In [13]:
structed_output.email

'chulsoo.kim@bikecorporation.me'

In [14]:
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요."
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수 상무가 ZENESIS 자전거 유통 협력 논의를 위해 다음 주 화요일 오전 10시에 귀사 사무실에서 미팅을 진행하자고 제안했습니다.', date='1월 15일 오전 10시')

In [15]:
llm = ChatOpenAI(
    temperature=0, model="gpt-5.6-luna"
)

llm.invoke("대한민국의 수도는 뭐야?")

AIMessage(content='대한민국의 수도는 **서울특별시**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 14, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EOYni0UG3uFaW5b7AAe1LQLlo4gTw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a7d8-1d03-7a80-ad1a-bb67affbd877-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 16, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
llm_with_structed = ChatOpenAI(
    temperature=0, model="gpt-5.6-luna"
).with_structured_output(EmailSummary)  # with_structured_output은 stream 지원을 하지 않음.

In [17]:
answer = llm_with_structed.invoke(email_conversation)
answer.person

'김철수'